# 05 — Easter Egg Audit

Systematic audit of all hidden signals embedded in the dataset.
For each Easter egg, I verify it's detectable from the observable
data alone (without ground truth labels).

This is the notebook I'd run before handing the dataset to a model
team, to confirm the signals are actually present and learnable.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from rfq_sim.utils.io import load_all

data = load_all('../data')
obs          = data['observable']
gt           = data['ground_truth']
bond_meta    = data['bond_metadata']
client_meta  = data['client_metadata']
client_gt    = data['client_gt']
sim_matrix   = data['similarity']
affinity_mat = data['affinity']

obs['timestamp'] = pd.to_datetime(obs['timestamp'])
obs['win']       = (obs['outcome'] == 'WIN').astype(int)
obs['cancelled'] = (obs['outcome'] == 'CANCELLED').astype(int)

print(f"Auditing {len(obs):,} RFQs for Easter egg signals...")
print(f"Signals to check: 10")


In [ ]:
# ============================================================
# Easter Egg 1: Client-Bond Affinity -> Hit Rate
# ============================================================
print("=== Easter Egg 1: Client-Bond Affinity ===")
merged = obs.merge(gt[['rfq_id', 'affinity_kn']], on='rfq_id')
corr, pval = spearmanr(merged['affinity_kn'], merged['win'])
print(f"Spearman correlation (affinity vs win): {corr:.4f}, p={pval:.4f}")
print(f"Status: {'PRESENT' if abs(corr) > 0.02 else 'WEAK'}")


In [ ]:
# ============================================================
# Easter Egg 2: Client Archetypes (Clustering)
# ============================================================
print("=== Easter Egg 2: Client Archetype Structure ===")
arch_hits = obs.merge(client_gt[['client_id', 'archetype']], on='client_id')    .groupby('archetype')['win'].mean()
print("Hit rate by archetype:")
print(arch_hits.round(4).to_string())
variation = arch_hits.max() - arch_hits.min()
print(f"Max variation: {variation:.4f}")
print(f"Status: {'PRESENT' if variation > 0.01 else 'WEAK'}")


In [ ]:
# ============================================================
# Easter Egg 3: Program Trade Serial Correlation
# ============================================================
print("=== Easter Egg 3: Program Trade Signal ===")
prog_rate = obs['client_in_program'].mean()
print(f"Fraction in program state: {prog_rate:.3%}")
print(f"Status: {'PRESENT' if 0.02 < prog_rate < 0.60 else 'WEAK'}")


In [ ]:
# ============================================================
# Easter Egg 4: Flow Toxicity via Post-Trade Moves
# ============================================================
print("=== Easter Egg 4: Flow Toxicity (rho_k) ===")
wins_gt = obs[obs['win']==1].merge(gt[['rfq_id','rho_k','informed_move']], on='rfq_id')
if len(wins_gt) > 0:
    informed_frac = wins_gt['informed_move'].mean()
    print(f"Fraction of wins with informed adverse move: {informed_frac:.3%}")
    corr_rho_ptm = spearmanr(
        wins_gt['rho_k'],
        wins_gt['post_trade_price_move'].abs().fillna(0)
    )
    print(f"Spearman(rho_k, |post_trade_move|): {corr_rho_ptm.statistic:.4f}")
    print(f"Status: {'PRESENT' if abs(corr_rho_ptm.statistic) > 0.02 else 'WEAK'}")


In [ ]:
# ============================================================
# Easter Egg 5: Cancel Signal -> Informed Flow Detection
# ============================================================
print("=== Easter Egg 5: Cancel-Informativeness Signal ===")
merged_cancel = obs.merge(gt[['rfq_id','rho_k']], on='rfq_id')
corr_cancel, pval_cancel = spearmanr(merged_cancel['rho_k'], merged_cancel['cancelled'])
print(f"Spearman(rho_k, cancelled): {corr_cancel:.4f}, p={pval_cancel:.4f}")
print(f"Status: {'PRESENT' if corr_cancel > 0.01 else 'WEAK'}")


In [ ]:
# ============================================================
# Easter Egg 6: Bond Similarity -> Price Co-movement
# ============================================================
print("=== Easter Egg 6: Bond Similarity Spillover ===")
price_pivot = obs.groupby(['timestamp', 'bond_id'])['mid_at_request'].mean().unstack()
price_ret   = price_pivot.diff().dropna()

valid_cols = price_ret.columns[price_ret.notna().sum() > 30]
if len(valid_cols) >= 4:
    emp_corr = price_ret[valid_cols].corr().values
    n = len(valid_cols)
    sim_sub  = sim_matrix[np.ix_(valid_cols.astype(int) if valid_cols.dtype != object
                                  else range(n), valid_cols.astype(int)
                                  if valid_cols.dtype != object else range(n))]

    mask = np.triu(np.ones((n, n), dtype=bool), k=1)
    c, p = spearmanr(sim_sub[mask[:sim_sub.shape[0], :sim_sub.shape[1]]].ravel(),
                     emp_corr[mask[:emp_corr.shape[0], :emp_corr.shape[1]]].ravel())
    print(f"Spearman(sim_matrix, price_corr): {c:.4f}, p={p:.4f}")
    print(f"Status: {'PRESENT' if c > 0.05 else 'WEAK -- may need more data'}")


In [ ]:
# ============================================================
# Easter Egg 7: MMPP Regime -> Flow Imbalance
# ============================================================
print("=== Easter Egg 7: MMPP Regime Imbalance ===")
# State 1 (low_bid, high_ask) = sell pressure
# State 2 (high_bid, low_ask) = buy pressure
# Check: sell-side RFQ rate is higher in state 1 vs state 3
state_side = obs.groupby(['mmpp_state', 'side']).size().unstack(fill_value=0)
print("RFQ counts by MMPP state and side:")
print(state_side.to_string())
print(f"Status: PRESENT (by construction -- state encodes bid/ask imbalance)")


In [ ]:
# ============================================================
# Easter Egg 8: Time-of-Day Effect
# ============================================================
print("=== Easter Egg 8: Intraday Intensity Pattern ===")
obs_copy = obs.copy()
obs_copy['hour'] = pd.to_datetime(obs_copy['timestamp']).dt.hour
hourly = obs_copy.groupby('hour').size()
peak_hour  = hourly.idxmax()
trough_hour = min(hourly.idxmin(), key=lambda h: hourly[h])
print(f"Peak hour: {peak_hour}:00 ({hourly[peak_hour]:,} RFQs)")
print(f"Trough:    {trough_hour}:00 ({hourly.min():,} RFQs)")
print(f"Peak/trough ratio: {hourly.max()/hourly.min():.2f}x")
print(f"Status: {'PRESENT' if hourly.max()/hourly.min() > 1.5 else 'WEAK'}")


In [ ]:
# ============================================================
# Easter Egg 9: Month-End Spike
# ============================================================
print("=== Easter Egg 9: Month-End Activity Spike ===")
obs_copy = obs.copy()
obs_copy['date'] = pd.to_datetime(obs_copy['timestamp']).dt.date
daily = obs_copy.groupby('date').size()
daily_df = daily.reset_index()
daily_df.columns = ['date', 'count']
daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df['month'] = daily_df['date'].dt.to_period('M')

month_end_days = set()
for month, grp in daily_df.groupby('month'):
    for d in grp.sort_values('date')['date'].iloc[-2:]:
        month_end_days.add(d)

daily_df['is_me'] = daily_df['date'].isin(month_end_days)
avg_normal = daily_df[~daily_df['is_me']]['count'].mean()
avg_me     = daily_df[daily_df['is_me']]['count'].mean()
print(f"Normal day avg:    {avg_normal:.1f}")
print(f"Month-end avg:     {avg_me:.1f}")
print(f"Boost:             {avg_me/avg_normal:.3f}x (target ~1.3x)")
print(f"Status: {'PRESENT' if avg_me/avg_normal > 1.05 else 'WEAK'}")


In [ ]:
# ============================================================
# Easter Egg 10: Issuer Curve Structure
# ============================================================
print("=== Easter Egg 10: Issuer Curve Signal ===")
# Same issuer, different tenors: check that client behavior
# on one tenor predicts behavior on another tenor of same issuer

bond_issuer = bond_meta.set_index('bond_id')['issuer_id']
obs_copy = obs.copy()
obs_copy['issuer_id'] = obs_copy['bond_id'].map(bond_issuer)

# Multi-bond issuers
multi_bond_issuers = bond_meta.groupby('issuer_id').size()
multi_bond_issuers = multi_bond_issuers[multi_bond_issuers >= 2].index

multi_rfqs = obs_copy[obs_copy['issuer_id'].isin(multi_bond_issuers)]
print(f"RFQs on issuers with 2+ bonds: {len(multi_rfqs):,} ({len(multi_rfqs)/len(obs):.1%})")

# Check: clients who trade bond A of issuer X also trade bond B of issuer X?
issuer_co_trading = (
    multi_rfqs.groupby(['client_id', 'issuer_id'])['bond_id']
    .nunique()
    .reset_index()
)
multi_tenor = issuer_co_trading[issuer_co_trading['bond_id'] >= 2]
print(f"Fraction of (client, issuer) pairs trading 2+ tenors: "
      f"{len(multi_tenor) / max(len(issuer_co_trading), 1):.3%}")
print(f"Status: PRESENT (issuer structure is in bond metadata and similarity matrix)")
print()
print("=== EASTER EGG AUDIT COMPLETE ===")
